# Day 030 · 第一阶段总结 + 学习路线图 · 中国版
**Phase 1 Recap** · 阶段 P1 · 量化基础

> 今天是 P1 阶段的最后一节,也是承上启下的一节。前 29 天我们从 0 到 1 把量化金融的「地基」打完了 — 你已经懂数据从哪里来、懂基本的统计语言、懂股债期权加密这几大类资产、懂交易所撮合到行情频率这一套市场结构、最后还亲手画过一份完整的策略报告。这一节不讲新东西,我们把前 29 天的所有内容做三件事 — 第一,用一张「四大支柱」的知识图谱把所有概念串起来,让你看清楚每节课在大图里的位置;第二,给你一份「30 个核心概念」的自检清单,挑出你已经掌握的、还需要回去补的;第三,告诉你 P2 阶段(D31-D60)接下来 4 周要学什么,从 Anaconda 环境搭建开始,我们要把工具栈从入门变成真正干活的程度。学完这一节你就完成了 P1,可以理直气壮跟别人说一句「我懂量化基础了」。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-20  ·  **建议学习时长:** 18 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [1]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


✓ 所有 8 个必需库已就绪
✓ 中文字体已加载:NotoSansCJK-Regular.ttc
✓ 环境就绪


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [2]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据


## 学习目标

- 建立量化金融四大支柱的知识图谱 — 数据 / 统计 / 资产 / 市场结构,每个支柱下哪几节课在讲什么
- 拿到 30 个核心概念自检清单 — 涵盖收益率、Sharpe、回撤、止损、波动率、期权希腊字母、撮合、复权、L2 等所有 P1 关键概念
- 看清楚自己 P1 学习的「白点」 — 哪些概念已经熟、哪些还需要回去补、哪些可以暂时放一放
- 了解 P2 阶段(D31-D60)4 周的学习路线 — Anaconda + NumPy + Pandas + Matplotlib + Jupyter + 项目工程化,从基础工具到自动化流水线
- 学习一套「螺旋上升」的方法论 — 先广度再深度、先模仿再创造、先简单再复杂,这是所有量化老兵共同的成长曲线

## 历史背景:29 天的旅程 — 你已经走完了多少

我们回到 29 天前。那时候你打开第一节课,我说量化金融是「数据 + 模型 + 工程」三件事。当时这句话对你来说可能就是一句口号,不知道具体长什么样。29 天过去了,这句话在你脑子里应该已经长出血肉了。

数据这件事,你现在知道从哪里拉。第 26 课讲过国内三大数据接口 — yfinance 海外、akshare 国内零翻墙、Tushare 专业付费。每个接口字段不一样、复权方式不一样、连通稳定性不一样。如果有人现在让你拉一只 A 股的5 年日线,你能10 分钟内写出代码,这一关就过了。

模型这件事,你也不陌生了。第 9 课的 Beta 回归、第 10 课的因子模型、第 12 课的均值回归、第 17 课的 Black-Scholes、第 21 课的 Delta-Gamma 对冲、第 28 课的频率聚合 — 这些都是模型。你不一定能从头推导,但你知道每个模型解决什么问题、什么时候用、什么时候失效。一个模型最重要的不是数学优雅,而是「知道它什么时候是错的」。

工程这件事是 P2 阶段的重头戏。前 29 天我们用的工具是边学边搭,装包、跑代码、画图都是手动的。从下一节开始,我们要把这些环节工程化 — Anaconda 隔离环境、Jupyter 调试、版本控制、自动化测试、生产部署。工程能力是量化研究员跟金融分析师最大的差别,也是国内大多数「半量化半主观」公司没有真正量化能力的根本原因。

你这 29 天投入了大约 8 小时课程时间,加上自己跑代码、看 notebook、做笔记的时间,保守估计 30 小时左右。30 小时换来一个完整的量化基础认知框架,这个投入产出比在职业培训里算极高的。下一阶段我们继续往里走。今天先停下来,把 P1 完整复盘一遍,做到「心中有图」,然后再上路。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. 量化金融四大支柱 — P1 的知识地图

P1 29 节课看似零散,本质上围绕四大支柱展开。每一根柱子下面是几节课,每节课又是这根柱子的一个具体侧面。理解这张图,你才能知道自己在大图里的位置。

第一根柱子 — 数据。这是所有量化的起点,数据不靠谱后面什么都白搭。D1-D9 我们讲了数据基础(K 线、成交量、收益率、对数收益率、年化、复合)。D26-D27 讲数据来源(yfinance / akshare / Tushare)和复权处理。D28 讲频率(日线 / 分钟 / Tick / L2)。这些加起来回答一个问题 — 你怎么把世界上的市场行为变成 Python 里的一张表。

第二根柱子 — 统计。这是量化的数学语言。D9-D13 讲基础统计(标准差、协方差、回归、Beta、相关性、显著性、p 值)。D14-D16 讲时序统计(自相关、平稳性、单位根、ARMA、波动率聚集)。这些是描述市场行为的工具,所有的策略背后都用到这些概念。

第三根柱子 — 资产类别。这是量化的物理对象。D6-D8 股票基础(估值、市盈率、市净率、ROE)。D17-D21 期权(call/put、希腊字母、隐含波动率、对冲)。D17-D18 加密(BTC / ETH / DeFi)。每类资产有自己的特性 — 股票看公司、期权看波动率、加密看流动性。

第四根柱子 — 市场结构。这是量化的战场地形。D22-D25 讲订单簿(LOB / 流动性 / 价差 / 撮合机制)。D25 比较全球交易所差别。D28 讲行情频率与策略匹配。这一柱讲清楚「市场是怎么运转的」,这是高手跟新手最大的认知差别。

这四根柱子立起来,P1 阶段才算结实。任何一根缺一截,后面遇到坑必踩。今天的自检清单帮你看清楚每根柱子下你的覆盖度。


### 2. 30 个核心概念自检清单

下面这 30 个概念是 P1 阶段的「最低门槛」。如果某一个你说不清楚是什么、什么时候用、跟其他概念什么关系,就需要回去看对应的那节课。

数据类(8 个) — ① 简单收益率 vs 对数收益率(D3),② 年化、月化、日化收益(D3),③ 复利与累计收益(D3),④ K 线 OHLC 含义(D2),⑤ 复权(前复权 / 后复权 / 不复权,D27),⑥ 行情频率(日线 / 分钟 / Tick / L2,D28),⑦ 成交量 vs 成交额(D2),⑧ 数据缺失处理(D9 + D26)。

统计类(8 个) — ⑨ 均值 / 标准差 / 方差(D9),⑩ 协方差与相关性(D9-10),⑪ 1 元线性回归与 Beta(D9),⑫ R² 决定系数(D9),⑬ p 值与显著性(D13),⑭ Z-score 标准化(D12),⑮ 滚动统计与移动均值(D12),⑯ 时序自相关(D14)。

资产类(7 个) — ⑰ 股票估值 PE / PB / ROE(D7),⑱ ETF 跟踪误差(D6),⑲ 期权 call / put 损益图(D17),⑳ 期权希腊字母(Delta / Gamma / Theta / Vega,D18-D19),㉑ 隐含波动率(D18),㉒ Delta 对冲与 Gamma 修正(D20-D21),㉓ 加密永续合约资金费率(D17)。

市场结构类(7 个) — ㉔ 订单簿与 LO/MO(D22),㉕ 价差与冲击成本(D23),㉖ 价格优先 + 时间优先撮合(D22),㉗ FIFO vs Pro-rata 分配(D24),㉘ 集合竞价 vs 连续竞价(D24),㉙ 主流交易所对比(D25),㉚ 数据频率与策略匹配(D28)。

做法 — 拿一张纸,把 1 到 30 编号,自检每一个能不能用 1 分钟跟别人讲清楚。打勾的留着,打叉的马上回去看对应那节课。这套自检你做完一遍,P1 才算真闭环。


### 3. 白点诊断 — 哪些必须补,哪些可以放

30 个概念里不是每一个都同等重要。下面把它们按「必须立刻补 / 重要但不紧急 / 可以暂时放」三档分一下,帮你省时间。

必须立刻补(打叉了一定要回去看)— ① 收益率与累计收益(D3),③ 复利,⑨ 标准差与方差(D9),⑪ Beta 回归(D9),⑭ Z-score(D12),⑲ call/put 损益图(D17),⑳ 希腊字母四件套(D18-D19),㉔ LO vs MO(D22),㉖ 价格优先 + 时间优先(D22),⑤ 复权(D27)。这 10 个是 P1 的「绝对基础」,P2 P3 P4 任何一节都会用到,缺了基本走不动。

重要但不紧急(打叉了 P2 期间可以慢慢补)— ⑩ 协方差矩阵(D10),⑫ R² 决定系数(D9),⑮ 滚动统计(D12),⑯ 自相关(D14),⑰ PE / PB / ROE(D7),⑱ ETF 跟踪误差(D6),㉒ Delta-Gamma 对冲(D20-D21),㉕ 价差与冲击成本(D23),㉘ 集合竞价(D24),㉚ 数据频率与策略匹配(D28)。这 10 个是「重要进阶」,P2 期间会被反复用到,补的时候带着 P2 项目实战补效果更好。

可以暂时放(打叉了等具体项目再回来看)— ② 年化、月化的换算细节(D3),④ K 线高低位含义(D2),⑥ Tick vs L2 细节(D28),⑦ 成交量 vs 成交额(D2),⑧ 数据缺失处理(D9 + D26),⑬ 显著性细节(D13),㉑ 隐含波动率求解(D18),㉓ 资金费率(D17),㉗ FIFO vs Pro-rata 细节(D24),㉙ 跨交易所对比(D25)。这 10 个是「细节深化」,等你做某个具体项目需要时再回来看,提前死磕是浪费时间。

这套分类不是教条,是大量量化新手的踩坑经验。我自己看过几十个量化转行案例,跑得快的人都是「先抓核心 10 个,其他边做边补」。一上来就想 30 个全精通的,90% 半年以内放弃。


### 4. P2 阶段 4 周路线 — 工程化是分水岭

P2 阶段(D31-D60)接下来 4 周做一件事 — 工程化。把 P1 那些「能跑通的脚本」变成「能在生产环境里稳定运行的项目」。这是量化研究员跟金融分析师的根本差别,也是大多数国内「半量化」公司没真正做到的事。

第 1 周(D31-D35) — 环境与工具栈。从今天开始的下一节(D31),我们讲 Anaconda + 虚拟环境 + 包管理 + 镜像源,把你的工作环境从「装一个 Python 就开搞」升级到「每个项目独立环境、可复现、可分享」。D32 NumPy 入门、D33 NumPy 进阶、D34 Pandas 基础、D35 Pandas 高级。学完这周你能写出像样的数据处理代码。

第 2 周(D36-D40) — Pandas + 数据 IO 与 visualization。D36-D37 时间序列处理、D38 Pandas 性能优化(向量化、Categorical、分块)、D39 Matplotlib 入门、D40 Seaborn 与高级可视化。学完这周你能做漂亮的策略报告图表。

第 3 周(D41-D45) — Jupyter Notebook 工作流。D41 Jupyter Lab 配置、D42-D43 nbformat 与 nbconvert 自动化、D44 Papermill 参数化执行、D45 Voila 把 notebook 变成 web app。学完这周你能搭建一套自动化研究平台。

第 4 周(D46-D50) — 项目工程化。D46 Git 版本控制、D47 项目目录与 Cookiecutter 模板、D48 Pytest 测试与 CI、D49 Logging 与监控、D50 第二阶段实战项目(把 D29 的策略报告做成一个工程化项目)。

4 周下来你的工具栈从「跑得通」变成「跑得稳」,这是量化研究员的入门门槛。P2 学完你才算真正能给量化机构投简历了。


### 5. 螺旋上升方法论 — 量化老兵的成长曲线

学量化几年下来,我观察到一个非常稳定的规律 — 跑得最快的人都不是「按教材顺序学」的,而是「广度 → 深度 → 广度 → 深度」螺旋上升的。

第一圈 — 广度 1.0。把整个量化领域大致涵盖什么知道清楚,每个概念知道是什么、解决什么问题、用什么工具。不深究细节,目标是「建立完整地图」。这正是 P1 这 29 节课在做的事。学完你应该能用 5 分钟跟一个外行讲清楚量化是什么、跟主观投资什么区别、有哪些主要流派。

第二圈 — 深度 1.0。选 1-2 个你最感兴趣的方向,深挖到能跑出实战代码。比如对期权 Delta-Gamma 对冲特别感兴趣,就花一两个月把 Black-Scholes 推导、希腊字母代码实现、动态对冲回测、波动率曲面建模都做一遍。这一圈做完你应该能在某个细分方向给别人讲半小时。P2 + P3 阶段就是引导你做第二圈。

第三圈 — 广度 2.0。带着前两圈的认知重新回到大图,把之前看不懂的、跳过的、忽略的部分补上。这次的广度不是表面上的「知道有这个东西」,而是「知道这个东西在我自己的项目里能起什么作用」。P4 + P5 阶段是第三圈。

第四圈 — 深度 2.0。选你要长期做的方向,深挖到论文级别 / 生产级别 / 创业级别。这一圈可能要几年,大多数量化老兵在这一圈定型。P6 之后是这一圈。

这个螺旋的核心是「不要在第一圈追求深度」。我见过太多新手第一圈就死磕 Black-Scholes 推导,卡住3 个月放弃。也见过太多新手第二圈就想覆盖所有方向,最后什么都没深入。第一圈广 + 第二圈深 + 第三圈广 + 第四圈深,这才是稳的节奏。

你现在刚走完第一圈广度。不要急着选方向,先做 P2 工程化打基础,P3 P4 自然会出现你最感兴趣的方向。这一点是 P1 阶段最重要的方法论收获,比任何具体技术都珍贵。


## 实操:P1 知识图谱可视化 — 四大支柱 + 28 节覆盖 + 30 个核心概念分布(中国版 — 跟原版相同,无需联网)

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [3]:
# day_030_phase1_recap_map.py — P1 阶段总结 + 学习路线图可视化
# 三张图:四大支柱知识图谱 + 28 节内容分布 + 30 个核心概念覆盖热图
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import json
from pathlib import Path

# ============ 1. 读 P1 课程数据 ============
print('==== 1. 读取 P1 课程结构 ====')
# 读 curriculum.json,提取 D1-D29 标题
cur_path = Path('scripts/curriculum.json')
if cur_path.exists():
    cur = json.loads(cur_path.read_text(encoding='utf-8'))
    p1_lessons = [l for l in cur['lessons'] if 1 <= l['d'] <= 29]
else:
    p1_lessons = []
print(f'P1 阶段共 {len(p1_lessons)} 节课')

# 四大支柱归类(基于课程内容人工归类)
pillar_map = {
    '数据 (Data)': [1, 2, 3, 26, 27, 28],
    '统计 (Statistics)': [4, 5, 9, 10, 11, 12, 13, 14, 15, 16],
    '资产 (Assets)': [6, 7, 8, 17, 18, 19, 20, 21],
    '市场结构 (Market Structure)': [22, 23, 24, 25],
}
# D29 是实战,跨支柱;D30 是总结,不归类

# 30 个核心概念(分四大类)
concepts_30 = {
    '数据': ['简单收益率 vs 对数', '年化 / 月化 / 日化', '复利与累计收益', 'K 线 OHLC',
            '复权处理', '行情频率', '成交量 vs 成交额', '数据缺失处理'],
    '统计': ['均值 / 标准差', '协方差与相关性', '1 元回归 + Beta', 'R² 决定系数',
            'p 值与显著性', 'Z-score 标准化', '滚动统计', '时序自相关'],
    '资产': ['股票估值 PE/PB/ROE', 'ETF 跟踪误差', 'Call/Put 损益图', '希腊字母',
            '隐含波动率', 'Delta 对冲', '加密资金费率'],
    '市场结构': ['订单簿 LO/MO', '价差与冲击成本', '价格 + 时间优先', 'FIFO vs Pro-rata',
                '集合 vs 连续竞价', '主流交易所对比', '频率与策略匹配'],
}

# 难度评级(1=必修, 2=重要, 3=细节)
priority = {
    '简单收益率 vs 对数': 1, '复利与累计收益': 1, '均值 / 标准差': 1, '1 元回归 + Beta': 1,
    'Z-score 标准化': 1, 'Call/Put 损益图': 1, '希腊字母': 1, '订单簿 LO/MO': 1,
    '价格 + 时间优先': 1, '复权处理': 1,
    '协方差与相关性': 2, 'R² 决定系数': 2, '滚动统计': 2, '时序自相关': 2,
    '股票估值 PE/PB/ROE': 2, 'ETF 跟踪误差': 2, 'Delta 对冲': 2,
    '价差与冲击成本': 2, '集合 vs 连续竞价': 2, '频率与策略匹配': 2,
    '年化 / 月化 / 日化': 3, 'K 线 OHLC': 3, '行情频率': 3, '成交量 vs 成交额': 3,
    '数据缺失处理': 3, 'p 值与显著性': 3, '隐含波动率': 3, '加密资金费率': 3,
    'FIFO vs Pro-rata': 3, '主流交易所对比': 3,
}

# ============ 2. 画图 ============
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# 2-1. 四大支柱框架图
ax = axes[0, 0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('① 量化金融 P1 四大支柱', fontsize=14, fontweight='bold')
# 中心
ax.add_patch(patches.Circle((5, 5), 1.2, facecolor='#FF6B6B', edgecolor='black', linewidth=2))
ax.text(5, 5, 'P1\n量化基础', ha='center', va='center', fontsize=13, fontweight='bold', color='white')
# 四根支柱
pillars_pos = [
    ('数据\nData', 2, 8, '#4ECDC4'),
    ('统计\nStatistics', 8, 8, '#FFE66D'),
    ('资产\nAssets', 2, 2, '#95E1D3'),
    ('市场结构\nMarket', 8, 2, '#C589E8'),
]
for name, x, y, color in pillars_pos:
    ax.add_patch(patches.FancyBboxPatch((x-1, y-0.7), 2, 1.4, boxstyle='round,pad=0.1', facecolor=color, edgecolor='black', linewidth=1.5))
    ax.text(x, y, name, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.plot([5, x], [5, y], 'k--', alpha=0.4, linewidth=1)

# 2-2. 28 节课在四支柱的分布
ax = axes[0, 1]
pillars = list(pillar_map.keys())
counts = [len(v) for v in pillar_map.values()]
colors_p = ['#4ECDC4', '#FFE66D', '#95E1D3', '#C589E8']
bars = ax.barh(pillars, counts, color=colors_p, edgecolor='black')
for bar, c in zip(bars, counts):
    ax.text(c + 0.1, bar.get_y() + bar.get_height()/2, f'{c} 节', va='center', fontsize=11)
ax.set_xlabel('节数', fontsize=11)
ax.set_xlim(0, max(counts) * 1.2)
ax.set_title('② P1 28 节在四大支柱的分布', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# 2-3. 30 个核心概念分类堆叠柱状图
ax = axes[1, 0]
cats = list(concepts_30.keys())
p1_count = [sum(1 for c in concepts_30[k] if priority.get(c, 3) == 1) for k in cats]
p2_count = [sum(1 for c in concepts_30[k] if priority.get(c, 3) == 2) for k in cats]
p3_count = [sum(1 for c in concepts_30[k] if priority.get(c, 3) == 3) for k in cats]
x = np.arange(len(cats))
ax.bar(x, p1_count, label='必修(立刻补)', color='#FF6B6B', edgecolor='black')
ax.bar(x, p2_count, bottom=p1_count, label='重要(慢慢补)', color='#FFE66D', edgecolor='black')
ax.bar(x, p3_count, bottom=np.array(p1_count)+np.array(p2_count), label='细节(放一放)', color='#95E1D3', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(cats, fontsize=10)
ax.set_ylabel('概念数', fontsize=11)
ax.set_title('③ 30 个核心概念按优先级分类', fontsize=14, fontweight='bold')
ax.legend(loc='upper right'); ax.grid(axis='y', alpha=0.3)

# 2-4. P1 → P2 学习路线时间轴
ax = axes[1, 1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('④ P1 → P2 学习路线时间轴', fontsize=14, fontweight='bold')
phases = [
    ('P1 量化基础', 0.5, 8, 2, '#FF6B6B', 'D1-D30\n29 节,30 小时\n四大支柱'),
    ('P2 工程化', 2.7, 8, 2, '#FFE66D', 'D31-D60\n30 节\nNumPy/Pandas\nJupyter/Git'),
    ('P3 策略', 4.9, 8, 2, '#95E1D3', 'D61-D120\n60 节\n回测/因子\n机器学习'),
    ('P4 实盘', 7.1, 8, 2, '#C589E8', 'D121-D200\n80 节\n执行/风控\n监控/优化'),
]
for name, x, y, w, color, detail in phases:
    ax.add_patch(patches.FancyBboxPatch((x, y-0.6), w, 1.2, boxstyle='round,pad=0.05', facecolor=color, edgecolor='black', linewidth=1.5))
    ax.text(x + w/2, y, name, ha='center', va='center', fontsize=11, fontweight='bold')
    ax.text(x + w/2, y - 2.5, detail, ha='center', va='top', fontsize=9)
# 箭头
for i in range(len(phases) - 1):
    x0 = phases[i][1] + phases[i][3]
    x1 = phases[i+1][1]
    ax.annotate('', xy=(x1, 8), xytext=(x0, 8), arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
# 标 P1 完成位置
ax.text(1.5, 6, '★ 你在这里', ha='center', fontsize=12, color='red', fontweight='bold')
ax.annotate('', xy=(1.5, 7), xytext=(1.5, 6.3), arrowprops=dict(arrowstyle='->', color='red', lw=2))

plt.tight_layout()
plt.savefig('chart_01.png', dpi=120, bbox_inches='tight')
plt.close()
print('保存 chart_01.png')

# ============ 3. 打印自检清单 ============
print('\n==== 3. 30 个核心概念自检清单 ====')
for category, concepts in concepts_30.items():
    print(f'\n【{category}】({len(concepts)} 个)')
    for c in concepts:
        p = priority.get(c, 3)
        mark = '★' if p == 1 else ('☆' if p == 2 else '·')
        print(f'  {mark} {c}')
print('\n★ = 必修(立刻补) | ☆ = 重要(慢慢补) | · = 细节(放一放)')

# ============ 4. 打印学习路线 ============
print('\n==== 4. P2 阶段 4 周学习路线 ====')
weeks = [
    ('第 1 周 (D31-D35)', '环境与工具栈', 'Anaconda + NumPy + Pandas 基础'),
    ('第 2 周 (D36-D40)', 'Pandas 高级 + 可视化', '时间序列 + Matplotlib + Seaborn'),
    ('第 3 周 (D41-D45)', 'Jupyter 工作流', 'Lab + nbformat + Papermill + Voila'),
    ('第 4 周 (D46-D50)', '项目工程化', 'Git + 目录 + 测试 + Logging + 实战项目'),
]
for w, theme, content in weeks:
    print(f'  {w} | {theme:12s} | {content}')

print('\n==== P1 收官 — 进 P2 工程化 ====')
print('你的认知地图已经建立,接下来 4 周把工具栈练扎实,P3 才能真正动手做策略。')

==== 1. 读取 P1 课程结构 ====
P1 阶段共 0 节课
保存 chart_01.png

==== 3. 30 个核心概念自检清单 ====

【数据】(8 个)
  ★ 简单收益率 vs 对数
  · 年化 / 月化 / 日化
  ★ 复利与累计收益
  · K 线 OHLC
  ★ 复权处理
  · 行情频率
  · 成交量 vs 成交额
  · 数据缺失处理

【统计】(8 个)
  ★ 均值 / 标准差
  ☆ 协方差与相关性
  ★ 1 元回归 + Beta
  ☆ R² 决定系数
  · p 值与显著性
  ★ Z-score 标准化
  ☆ 滚动统计
  ☆ 时序自相关

【资产】(7 个)
  ☆ 股票估值 PE/PB/ROE
  ☆ ETF 跟踪误差
  ★ Call/Put 损益图
  ★ 希腊字母
  · 隐含波动率
  ☆ Delta 对冲
  · 加密资金费率

【市场结构】(7 个)
  ★ 订单簿 LO/MO
  ☆ 价差与冲击成本
  ★ 价格 + 时间优先
  · FIFO vs Pro-rata
  ☆ 集合 vs 连续竞价
  · 主流交易所对比
  ☆ 频率与策略匹配

★ = 必修(立刻补) | ☆ = 重要(慢慢补) | · = 细节(放一放)

==== 4. P2 阶段 4 周学习路线 ====
  第 1 周 (D31-D35) | 环境与工具栈       | Anaconda + NumPy + Pandas 基础
  第 2 周 (D36-D40) | Pandas 高级 + 可视化 | 时间序列 + Matplotlib + Seaborn
  第 3 周 (D41-D45) | Jupyter 工作流  | Lab + nbformat + Papermill + Voila
  第 4 周 (D46-D50) | 项目工程化        | Git + 目录 + 测试 + Logging + 实战项目

==== P1 收官 — 进 P2 工程化 ====
你的认知地图已经建立,接下来 4 周把工具栈练扎实,P3 才能真正动手做策略。


## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 量化转行案例追踪 | 国内某知名量化招聘平台数据 | 国内某量化招聘平台过去 5 年统计了 200+ 量化转行成功案例,发现一个非常一致的规律 — 跑得最快的人不是 985 数学专业出身,而是「先广度后深度」按部就班走过来的。其中有一位非常典型的案例 — 一位 40 岁的传统软件工程师,完全不懂金融,通过自学量化基础(类似今天 P1 这套体系)+ Python 进阶 + 一个 GitHub 量化项目,在 11 个月内成功转入某中型私募。他后来分享时反复强调「不要一开始就死磕 Black-Scholes,先把整个图谱建起来」 — 跟今天 P1 总结的思路完全一致。 |
| 国内私募招聘标准 | 幻方 / 九坤 / 灵均等头部 | 国内头部量化私募(幻方、九坤、灵均、明汯等)校招对量化研究员的笔试,基本覆盖 P1 涉及的所有概念 — 计算 Sharpe、画 K 线、回归算 Beta、解读期权希腊字母、识别撮合规则差异等。一位幻方面试官公开分享过:「我们筛简历看不出真本事,笔试一道 Beta 回归题就能筛掉 70% 候选人。」 P1 自检清单上的 30 个概念,正好是这种笔试题的概念库。这就是为什么 P1 不是「入门玩具」,而是真实门槛。 |
| GitHub 量化项目热门排行 | QuantLib / Backtrader / vnpy | GitHub 上最热门的几个量化开源项目 — QuantLib(C++ 期权定价库,4k+ star)、Backtrader(Python 回测框架,12k+ star)、vnpy(中国本土期货量化平台,20k+ star)。这些项目都建立在 P1 阶段的核心概念之上 — 你要看懂 Backtrader 源码,需要懂 K 线、收益率、Sharpe、回撤;要看懂 QuantLib 源码,需要懂希腊字母、隐含波动率;要看懂 vnpy 源码,需要懂订单簿、撮合规则。P1 学完你已经能看懂这些项目的 README,P2 之后你能看懂源码,P3 之后你能给这些项目提 PR。 |
| 知乎量化话题统计 | 知乎「量化交易」话题 | 知乎「量化交易」话题下问题数量统计,大约 80% 的「新手提问」都集中在 P1 阶段的 30 个核心概念上 — 「Sharpe 多少算好?」「Beta 怎么算?」「期权 Theta 是什么?」「为什么撮合是价格优先?」。这意味着如果你 P1 学扎实,你的认知水平已经超过了知乎 80% 自称「量化爱好者」的提问者。但反过来也说明 — P1 之外的真正进阶知识(策略实现、风险管理、组合优化、机器学习)在知乎上提问者很少,因为「真懂的人都在搬砖、自学的人还没到这一关」。P2 P3 才是真正分水岭。 |
| 在校学生量化竞赛 | CFRM / WorldQuant Brain / Optiver | 国际几个面向学生的量化竞赛 — WorldQuant Brain Quantathon(美国)、Optiver World Trader Championship(欧洲)、CFRM 量化竞赛(华盛顿大学)等。参赛门槛基本是 P1 全部知识 + 一个能跑通的 alpha 策略。这些竞赛获奖者大约 30% 收到顶级机构 offer。意思是 — 你如果在 P3 阶段(D90 之前)能跑通一个像样的 alpha 策略并参加这类竞赛,有一定概率拿到顶级机构 offer。这不是天方夜谭,是有真实路径的。但前提是 P1 P2 基础必须扎实,后面才有空间。 |


## 常见坑

### ⚠ 01. 急于「实盘验证」跳过 P2 工程化

学完 P1 最常见的冲动 — 想马上拿 D29 那份策略报告去实盘验证。但 P1 的代码都是教学版本,没考虑生产环境的延迟、监控、风控、断网恢复、订单管理等。实盘上手必出问题。**正确做法**:先把 P2 工程化走完,把你的策略变成一个可以可靠运行的项目,再考虑用真金白银验证。P1 后实盘的人,30 天内亏掉本金 20% 是常态。

### ⚠ 02. 30 个概念全想精通,陷入完美主义

另一种新手 trap — 想把 30 个概念每一个都精通到能跟人讲 30 分钟才往下走。这种思路下你 P2 阶段都开始不了,因为 30 个里总有一两个永远「还差一点」。**正确做法**:把核心 10 个(必须立刻补类)做到能讲 1 分钟,其余 20 个边做边补就行。完美主义是新手最大的敌人。

### ⚠ 03. 看了课没动手跑代码

P1 阶段大约 60% 的人没有真正动手跑 notebook,只是看视频做笔记。这种「被动学习」记忆 7 天后保留率只有 10%,跟没学过没区别。**正确做法**:从今天起每节课都必须把 notebook 跑一遍,每节都自己改 1-2 个参数看输出变化。手感比知识本身更重要。我们的 notebook 都有「中国版」零翻墙版本,没借口不跑。

### ⚠ 04. 选错方向 — 个人散户去模仿机构高频

P1 阶段最容易让人产生的幻觉 — 看了第 25 课交易所对比、第 28 课频率分析,觉得「我也可以做高频做市」。但高频做市需要的资金、基础设施、团队规模根本不是散户能玩的。**正确做法**:个人散户的最优起点永远是「中频量价 + 多因子」,这条路投入产出比最高。P3 P4 才会引导你做策略选择,P1 阶段不要急着选方向。

### ⚠ 05. 看了别人晒收益就「all-in」

P1 期间最大的诱惑 — 看到 B 站某博主晒「年化 300%」就一头扎进去模仿。十有八九博主是 cherry-picking 周期 + 没扣手续费 + 没披露最大回撤 + 在小资金上跑出的不可复制结果。**正确做法**:任何看到的「神奇收益」先问 4 个问题 — 时间区间多长?最大回撤多少?有没有扣手续费?资金规模能扩展到多少?这 4 个问题大多数晒收益的人答不上来,答上来的也基本是泡沫。这条认知能帮你 P2 期间避开 90% 的坑。

## 实战 SOP · P1 收官 + 进 P2 的 7 条总结

1. 完成 30 个核心概念自检 — 打叉的 10 个必修类必须回去看,其他可以慢慢补
2. 正式开始用「四大支柱」组织你的笔记 — 数据 / 统计 / 资产 / 市场结构,每个新概念归类到对应支柱下
3. 进 P2 前先确定你的工作环境 — 一台稳定的电脑、足够硬盘、稳定网络、注册好 Tushare / 国内数据账号
4. P2 阶段每节都必须动手跑 notebook — 看视频 + 跑代码 + 改参数三件套,缺一不可
5. 实盘验证至少推迟到 P3 完成之后 — 工程化没走完,真金白银必亏
6. 学习方法采用螺旋上升 — 先广度后深度,不要在 P1 就死磕某个具体方向
7. 每周复盘一次 — 这周学了什么、哪里不懂、下周补什么,这套循环跑稳了你的成长速度会肉眼可见

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. P1 阶段 29 节内容归到四大支柱 — 数据 / 统计 / 资产 / 市场结构,每根支柱下有 7-8 节课
3. 30 个核心概念自检清单 — 数据 8 个 + 统计 8 个 + 资产 7 个 + 市场结构 7 个,做完才算真正闭环
4. 白点诊断3 档 — 必须立刻补 10 个、重要但不紧急 10 个、可以暂时放 10 个,集中精力打核心
5. P2 阶段 4 周路线 — D31-D35 环境与 NumPy / Pandas、D36-D40 Pandas 与可视化、D41-D45 Jupyter 工作流、D46-D50 项目工程化
6. 工程化是量化研究员跟金融分析师的根本差别 — 也是国内大多数「半量化」公司没做到的事
7. 螺旋上升方法论 — 广度 1.0 → 深度 1.0 → 广度 2.0 → 深度 2.0,跑得最快的老兵都是这个节奏
8. P1 学完你的认知水平已经超过知乎 80% 量化爱好者 — 但 P2 P3 才是真正分水岭,不要止步不前

## 自测题

**Q1.** 量化金融的四大支柱是什么?各自回答什么问题?(提示:数据 = 怎么把市场变成 Python 里的表、统计 = 怎么描述市场、资产 = 物理对象、市场结构 = 战场地形)

**Q2.** 30 个核心概念里属于「必须立刻补」一档的有哪 10 个?为什么这 10 个最重要?(提示:它们是 P2-P6 反复用到的工具,缺了走不动)

**Q3.** P2 阶段 4 周分别学什么?为什么工程化是量化研究员的分水岭?(提示:NumPy/Pandas → 可视化 → Jupyter → 项目工程化;主观分析师不会工程化)

**Q4.** 为什么不建议 P1 学完立刻实盘?需要等到什么阶段才合适?(提示:P1 是教学代码,缺生产环境处理,至少 P3 完成 + 一个工程化项目跑通后再考虑)

**Q5.** 螺旋上升方法论的四圈是什么?为什么第一圈不应该追求深度?(提示:广度 1.0 → 深度 1.0 → 广度 2.0 → 深度 2.0;没建立大图直接深挖容易迷路放弃)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 031 · 环境搭建:Anaconda 工作流** (Setup)

第 31 课 — Anaconda 工作流环境搭建。P2 第一节,我们把工作环境从「装一个 Python 就开搞」升级到「每个项目独立环境、可复现、可分享」。讲 conda 虚拟环境、必装包清单、镜像源加速、Jupyter Lab vs VS Code 怎么选、目录结构怎么布。这是工程化的第一步,基础不打好后面 P2 后续 19 节会一直踩坑。

## 推荐阅读

- Marcos Lopez de Prado《Advances in Financial Machine Learning》(2018,Wiley)— 量化机器学习开山之作,但需要 P1-P5 全部基础才能读懂,先收藏 P6 阶段再啃
- Ernest Chan《Quantitative Trading》(2008 + 2013 续作,Wiley)— 散户友好的量化入门,P1 学完直接读,大约 200 页系统梳理整个量化流程
- Andreas Clenow《Following the Trend》(2013,Wiley)— 趋势跟踪策略的实战手册,P2 阶段读最合适,讲透 SMA 类策略的「真实可行性」
- Wes McKinney《Python for Data Analysis》(2022 第 3 版,O'Reilly)— Pandas 作者亲自写,P2 阶段的标准参考书,涵盖 NumPy + Pandas + Matplotlib 全套
- Python 工具栈进阶 — quantstats(策略报告生成)、empyrical(指标计算)、ffn(指标 + 可视化一体)、pyfolio(机构级风险归因报告);这四个库 P2 期间会反复用到,提前 pip install 备着